# **Selinum pour automatiser l’ouverture de pages TIKTOK**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install selenium pandas
!pip install openai-whisper
!pip install tqdm
# Mettre à jour et installer Chrome + Driver
!apt-get update -qq
!apt-get install -y google-chrome-stable
!apt-get install -y chromium-driver

# Installer les librairies Python
!pip install selenium webdriver-manager



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.0/512.0 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 16.7 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 54.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=b984270ee0917e9eac902dac1a34e67a5fe14c9bcd22cc4a3d82327de2949df9
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper

In [ ]:
!which google-chrome-stable
!google-chrome-stable --version


/bin/bash: line 1: google-chrome-stable: command not found


In [ ]:
# Télécharger la dernière version stable de Google Chrome
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb

# Installer le .deb avec dpkg
!dpkg -i google-chrome-stable_current_amd64.deb

# Corriger les dépendances si besoin
!apt-get -f install -y


Selecting previously unselected package google-chrome-stable.
(Reading database ... 122152 files and directories currently installed.)
Preparing to unpack google-chrome-stable_current_amd64.deb ...
Unpacking google-chrome-stable (144.0.7559.132-1) ...
dpkg: dependency problems prevent configuration of google-chrome-stable:
 google-chrome-stable depends on libatk-bridge2.0-0 (>= 2.5.3); however:
  Package libatk-bridge2.0-0 is not installed.
 google-chrome-stable depends on libatk1.0-0 (>= 2.11.90); however:
  Package libatk1.0-0 is not installed.
 google-chrome-stable depends on libatspi2.0-0 (>= 2.9.90); however:
  Package libatspi2.0-0 is not installed.
 google-chrome-stable depends on libvulkan1; however:
  Package libvulkan1 is not installed.
 google-chrome-stable depends on libxcomposite1 (>= 1:0.4.4-1); however:
  Package libxcomposite1 is not installed.

dpkg: error processing package google-chrome-stable (--install):
 dependency problems - leaving unconfigured
Processing trigge

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

# Options headless pour Colab
opts = Options()
opts.binary_location = "/usr/bin/google-chrome-stable"
opts.add_argument("--headless=new")
opts.add_argument("--no-sandbox")
opts.add_argument("--disable-dev-shm-usage")

# Lancer Chrome
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=opts
)

driver.get("https://www.google.com")
print("Titre de la page :", driver.title)
driver.quit()



Titre de la page : Google


In [ ]:
def download_tiktok_audio(url, idx=None):
    try:
        driver.get(url)
        time.sleep(5)  # attendre que la page charge

        # Récupérer l'élément audio
        audio_elem = driver.find_element(By.TAG_NAME, "audio")
        audio_url = audio_elem.get_attribute("src")

        if audio_url:
            import requests
            audio_path = os.path.join(audio_dir, f"audio_{idx}.mp3")
            r = requests.get(audio_url)
            with open(audio_path, "wb") as f:
                f.write(r.content)
            print(f"[{idx}] Audio téléchargé : {audio_path}")
            return audio_path
        else:
            print(f"[{idx}] Pas d'audio trouvé pour {url}")
            return None
    except Exception as e:
        print(f"[{idx}] Erreur pour {url} : {e}")
        return None


# **Yt-dlp extraire l’audio**

In [ ]:
!pip install yt-dlp


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.0/182.0 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 125.8 MB/s eta 0:00:00


In [ ]:
import yt_dlp
import os

def download_tiktok_audio(url, idx=0, output_dir="audios", cookies="all_cookies.txt"):
    os.makedirs(output_dir, exist_ok=True)
    file_path = os.path.join(output_dir, f"tiktok_{idx}.mp3")

    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': file_path,
        'extractaudio': True,
        'audioformat': 'mp3',
        'quiet': True,
        'cookiefile': cookies  # permet de gérer les vidéos privées/bloquées
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        return file_path
    except Exception as e:
        # print(f"[{idx}] Erreur avec {url}: {e}")
        return None


In [ ]:
from google.colab import files

uploaded = files.upload()


Saving dataset_tiktok-video-comment-scraper-task_2026-02-06_09-12-57-213.csv to dataset_tiktok-video-comment-scraper-task_2026-02-06_09-12-57-213.csv


In [ ]:
import pandas as pd

path = "/content/dataset_tiktok-video-comment-scraper.csv"

df = pd.read_csv(
    path,
    sep=None,              # autodétection du séparateur
    engine="python",
    encoding="utf-8",
    on_bad_lines="warn",
    quotechar='"',
    escapechar='\\'
)

print(df.shape)
df.head()


(200, 15)


,"﻿""url""",author_nickname,video_title,video_post_date,video_hashtags,video_comments,video_likes,video_views,video_shares,video_bookmarks,video_duration_seconds,comment_text,commenter_username,comment_likes,comment_date
0,https://www.tiktok.com/@dairasaenz_/video/7502...,Daira Saenz | Travel & Sydney,@Happy Travels I literally love you!!! Best tr...,2025-05-09 13:18:26,# #travelaustralia #happytravels #eastcoastaus...,7,40,2861,8,8,108,so exciteddd!!! ya quieroo😍😍😍,aleam00,1,2025-05-11 05:49:21
1,https://www.tiktok.com/@dairasaenz_/video/7502...,Daira Saenz | Travel & Sydney,@Happy Travels I literally love you!!! Best tr...,2025-05-09 13:18:26,# #travelaustralia #happytravels #eastcoastaus...,7,40,2861,8,8,108,We love this! So excited for your trip 🤩🌴✨,happytravelsoz,1,2025-05-11 23:33:24
2,https://www.tiktok.com/@dairasaenz_/video/7502...,Daira Saenz | Travel & Sydney,@Happy Travels I literally love you!!! Best tr...,2025-05-09 13:18:26,# #travelaustralia #happytravels #eastcoastaus...,7,40,2861,8,8,108,Thank you!,user9257655022329,1,2025-05-09 14:38:15
3,https://www.tiktok.com/@dairasaenz_/video/7502...,Daira Saenz | Travel & Sydney,@Happy Travels I literally love you!!! Best tr...,2025-05-09 13:18:26,# #travelaustralia #happytravels #eastcoastaus...,7,40,2861,8,8,108,We ❤️❤️this! So excited for your trip🤩🌴✨,Michaëlle Matali,1,2025-05-14 21:51:18
4,https://www.tiktok.com/@luxeinmonaco/video/751...,LUXEINMONACO,Stylish and brutal Billionaires in Monaco #lux...,2025-06-19 10:43:26,#luxurylifestyle #billionaires #monaco #richli...,3740,369900,11900000,11600,21180,64,The first guy gentleman,Oula monzer,73,2025-06-21 16:05:33


In [ ]:
from tqdm import tqdm

# sortie
AUDIO_DIR = "audios_tiktok"
os.makedirs(AUDIO_DIR, exist_ok=True)

#Fonction de téléchargement (Utilise yt-dlp)
def download_tiktok_audio(url, idx):
    try:
        filename = f"audio_{idx}"
        ydl_opts = {
            'format': 'bestaudio/best',
            'outtmpl': f'{AUDIO_DIR}/{filename}.%(ext)s',
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'mp3',
                'preferredquality': '192',
            }],
            'quiet': True,
            'no_warnings': True,
        }
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        return f"{AUDIO_DIR}/{filename}.mp3"
    except Exception:
        return None



# Nettoyage des noms de colonnes
df.columns = df.columns.str.strip().str.replace('"', '').str.replace('\ufeff', '')

# Renommer la colonne URL si nécessaire
if '' in df.columns:
    df.rename(columns={'': 'url'}, inplace=True)

# Extraction des URLs uniques pour éviter les téléchargements inutiles
urls_uniques = df['url'].dropna().drop_duplicates()

# Téléchargement une seule fois par vidéo
url2file = {}
for i, u in enumerate(tqdm(urls_uniques, desc="Téléchargements uniques")):
    url2file[u] = download_tiktok_audio(u, idx=i)

# Réplication du chemin du fichier sur toutes les lignes (commentaires)
df['audio_file'] = df['url'].map(url2file)

# Sauvegarde du nouveau CSV
df.to_csv("dataset_final_audio.csv", index=False, sep=';')

# Aperçu
print(df[['url', 'audio_file']].head())

Téléchargements uniques:   0%|          | 0/20 [00:00<?, ?it/s]

Téléchargements uniques:   5%|▌         | 1/20 [00:02<00:52,  2.78s/it]

Téléchargements uniques:  10%|█         | 2/20 [00:05<00:49,  2.74s/it]

Téléchargements uniques:  15%|█▌        | 3/20 [00:08<00:46,  2.71s/it]

Téléchargements uniques:  20%|██        | 4/20 [00:12<00:51,  3.21s/it]

...

Téléchargements uniques:  85%|████████▌ | 17/20 [00:49<00:08,  2.76s/it]

Téléchargements uniques:  90%|█████████ | 18/20 [00:51<00:05,  2.52s/it]

Téléchargements uniques:  95%|█████████▌| 19/20 [00:53<00:02,  2.33s/it]

Téléchargements uniques: 100%|██████████| 20/20 [00:54<00:00,  2.74s/it]

                                                 url  \
0  https://www.tiktok.com/@dairasaenz_/video/7502...   
1  https://www.tiktok.com/@dairasaenz_/video/7502...   
2  https://www.tiktok.com/@dairasaenz_/video/7502...   
3  https://www.tiktok.com/@dairasaenz_/video/7502...   
4  https://www.tiktok.com/@luxeinmonaco/video/751...   

                  audio_file  
0  audios_tiktok/audio_0.mp3  
1  audios_tiktok/audio_0.mp3  
2  audios_tiktok/audio_0.mp3  
3  audios_tiktok/audio_0.mp3  
4  audios_tiktok/audio_1.mp3  


# **WHIISPER pour transcrire des fichiers audio**

In [ ]:
!pip install openai-whisper
!pip install tqdm


In [ ]:
import whisper

# Charger le modèle
model = whisper.load_model("base")


In [ ]:
import os
import pandas as pd
from tqdm import tqdm
import whisper

csv_path = "/content/dataset_tiktok-video-comment-scraper-task_2026-02-06_09-12-57-213.csv"

# Transcrire une seule fois par fichier audio
uniq_files = df["audio_file"].dropna().drop_duplicates()
bn2txt = {}

for f in tqdm(uniq_files, desc="Transcription"):
    if not f or not os.path.exists(f):
        continue

    # Transcription initiale
    r = model.transcribe(f)
    txt = r["text"]

    # Traduction si la langue n'est pas l'anglais
    if r.get("language") != "en":
        txt = model.transcribe(f, task="translate")["text"]

    bn2txt[f] = txt

# Mettre la translation sur chaque ligne
df["Translation"] = df["audio_file"].map(lambda p: bn2txt.get(p, ""))

# Sauvegarde + aperçu
df.to_csv(csv_path, index=False)
print(df[["url", "audio_file", "Translation"]].head())

Transcription/Traduction: 100%|██████████| 20/20 [01:03<00:00,  3.18s/it]

                                                 url  \
0  https://www.tiktok.com/@dairasaenz_/video/7502...   
1  https://www.tiktok.com/@dairasaenz_/video/7502...   
2  https://www.tiktok.com/@dairasaenz_/video/7502...   
3  https://www.tiktok.com/@dairasaenz_/video/7502...   
4  https://www.tiktok.com/@luxeinmonaco/video/751...   

                  audio_file  \
0  audios_tiktok/audio_0.mp3   
1  audios_tiktok/audio_0.mp3   
2  audios_tiktok/audio_0.mp3   
3  audios_tiktok/audio_0.mp3   
4  audios_tiktok/audio_1.mp3   

                                         Translation  
0   I need to share this with you. I was literall...  
1   I need to share this with you. I was literall...  
2   I need to share this with you. I was literall...  
3   I need to share this with you. I was literall...  
4   I'm a man, I'm a man, I'm a man No I can't, I...  


In [ ]:
print("df rows:", len(df))
print("Translation:", len(df['Translation']))

df rows: 200
Translation: 200


# **ID pour les url**

In [ ]:
import pandas as pd
from pathlib import Path

#Définition du chemin de sortie
OUTPUT_PATH = '/content/dataset_tiktok-video-comment-scraper.csv'


df.columns = df.columns.str.strip().str.replace('"', '').str.replace('\ufeff', '')

url_col_name = None
if 'url' in df.columns:
    url_col_name = 'url'
elif 'video_url' in df.columns:
    url_col_name = 'video_url'
else:
    raise KeyError("Neither 'url' nor 'video_url' column found in DataFrame.")


df[url_col_name] = df[url_col_name].astype(str)

#Extraire les URLs uniques
unique_urls = df[url_col_name].dropna().drop_duplicates()

# Générer des IDs uniques

ids, _ = pd.factorize(unique_urls)
url_to_id_map = pd.Series(ids + 1129, index=unique_urls)

# mapping au DataFrame principal
df['video_id'] = df[url_col_name].map(url_to_id_map)

#Affichage pour vérification
print(f"Nombre de vidéos uniques détectées : {len(unique_urls)}")
print("\nAperçu des correspondances URL -> ID :")
print(df[[url_col_name, 'video_id']].drop_duplicates().head(10))

#Sauvegarde dans le fichier demandé
df.to_csv(OUTPUT_PATH, sep=';', index=False)

print(f"\n Fichier enregistré avec succès ici : {OUTPUT_PATH}")

Nombre de vidéos uniques détectées : 20

Aperçu des correspondances URL -> ID :
                                                  url  video_id
0   https://www.tiktok.com/@dairasaenz_/video/7502...      1129
4   https://www.tiktok.com/@luxeinmonaco/video/751...      1130
14  https://www.tiktok.com/@theschoolofhardknocks/...      1131
24  https://www.tiktok.com/@malaikahraja/video/740...      1132
34  https://www.tiktok.com/@missmargariita1/video/...      1133
44  https://www.tiktok.com/@harryjaggardtravel/vid...      1134
54  https://www.tiktok.com/@kaganbrooks/video/7429...      1135
63  https://www.tiktok.com/@emmas.travelling/video...      1136
70  https://www.tiktok.com/@divyadiscovers/video/7...      1137
80  https://www.tiktok.com/@refinedambition_/video...      1138

 Fichier enregistré avec succès ici : /content/dataset_tiktok-video-comment-scraper-task_2026-02-06_09-12-57-213.csv
